# Topic 2 Temperature Prediction MVP

This is a first notebook for the project idea. It only loads the data and does simple EDA. It does not build the final target or model yet.

## Simple Project Plan

1. Load the ERA5 temperature data and the climate index data.
2. Look at the time range, basic values, and simple plots.
3. Talk with the professor about a simple temperature outcome.
4. Later, join the temperature data with the climate indices.
5. Later, try a basic regression model and simple checks.

## Load Packages

In [ ]:
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

## Load Data

These files use different formats, so I read each one directly.

In [ ]:
with xr.open_dataset("data/ERA5_2mtemp_1x1.nc") as era5_file:
    era5_data = era5_file.load()

era5_data

In [ ]:
nino34_rows = []

with open("data/nina34.anom.data", "r") as nino34_file:
    for line in nino34_file:
        parts = line.split()
        if len(parts) >= 13 and parts[0].isdigit():
            year = int(parts[0])
            for month in range(1, 13):
                value = float(parts[month])
                if value in [-99.99, 99.99]:
                    value = None
                nino34_rows.append({"date": pd.Timestamp(year, month, 1), "nino34": value})

nino34_data = pd.DataFrame(nino34_rows)
nino34_data.head()

In [ ]:
pdo_rows = []

with open("data/ersst.v5.pdo.dat", "r") as pdo_file:
    for line in pdo_file:
        parts = line.split()
        if len(parts) >= 13 and parts[0].isdigit():
            year = int(parts[0])
            for month in range(1, 13):
                value = float(parts[month])
                pdo_rows.append({"date": pd.Timestamp(year, month, 1), "pdo": value})

pdo_data = pd.DataFrame(pdo_rows)
pdo_data.head()

In [ ]:
with open("data/ao.long.csv", "r") as ao_file:
    ao_data = pd.read_csv(ao_file)

ao_data.columns = ["date", "ao"]
ao_data["date"] = pd.to_datetime(ao_data["date"])
ao_data["ao"] = pd.to_numeric(ao_data["ao"], errors="coerce")
ao_data.loc[ao_data["ao"] == -999, "ao"] = None

ao_data.head()

## Quick Data Check

In [ ]:
print("ERA5 time:", str(era5_data.time.min().values)[:10], "to", str(era5_data.time.max().values)[:10])
print("ERA5 grid:", len(era5_data.lat), "lat values and", len(era5_data.lon), "lon values")
print("Nino 3.4 rows:", len(nino34_data))
print("PDO rows:", len(pdo_data))
print("AO rows:", len(ao_data))

In [ ]:
climate_index_data = nino34_data.merge(pdo_data, on="date", how="outer")
climate_index_data = climate_index_data.merge(ao_data, on="date", how="outer")
climate_index_data = climate_index_data.sort_values("date")

climate_index_data.head()

## Simple EDA

This section only gives a quick first look at the data.

In [ ]:
climate_index_data[["nino34", "pdo", "ao"]].describe()

In [ ]:
recent_climate_index_data = climate_index_data[climate_index_data["date"] >= "1980-01-01"]

plt.figure(figsize=(10, 4))
plt.plot(recent_climate_index_data["date"], recent_climate_index_data["nino34"], label="Nino 3.4")
plt.plot(recent_climate_index_data["date"], recent_climate_index_data["pdo"], label="PDO")
plt.plot(recent_climate_index_data["date"], recent_climate_index_data["ao"], label="AO")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Climate indices since 1980")
plt.xlabel("Date")
plt.ylabel("Index value")
plt.legend()
plt.show()

In [ ]:
temperature_name = list(era5_data.data_vars)[0]
temperature_data = era5_data[temperature_name]

temperature_data

In [ ]:
mean_temperature_celsius = temperature_data.mean(dim=["lat", "lon"]) - 273.15

plt.figure(figsize=(10, 4))
mean_temperature_celsius.plot()
plt.title("Simple mean ERA5 temperature")
plt.xlabel("Date")
plt.ylabel("Temperature (C)")
plt.show()

In [ ]:
first_month_temperature_celsius = temperature_data.isel(time=0) - 273.15

first_month_temperature_celsius.plot(figsize=(10, 4))
plt.title("ERA5 temperature in first month")
plt.show()

## Notes for Professor Meeting

- This is only a simple start.
- The data files can be loaded.
- The first EDA plots work.
- The next choice is the temperature outcome.
- After that, I can build a small model.